In [ ]:
# import pandas as pd

# df=pd.read_csv(
#     'MCC_product_lists/MCC_DataExport_esd-protection-devices(MCC-esd-protection-devices).csv',
#     header=0,
#     skiprows=[0,2]
# )

# df.columns = [' '.join(col.split()) for col in df.columns]

# # 单独去掉ESD0512LB，他有两种参数，如果一起放进df里会直接把float coerce成string
# df = df[df['Product'] != 'ESD0512LB']

# df_selected=df.iloc[:,[1,3]+list(range(4,17))]
# df_selected.info()

<class 'pandas.DataFrame'>
Index: 543 entries, 0 to 543
Data columns (total 15 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Product                                543 non-null    str    
 1   Compliance                             543 non-null    str    
 2   Number of Functions                    543 non-null    int64  
 3   Configuration                          543 non-null    str    
 4   Package Type                           543 non-null    str    
 5   Reverse Standoff Voltage VRWM(V)       543 non-null    str    
 6   Peak Pulse Current IPP(A)              543 non-null    float64
 7   Max. Clamping Voltage VC (V)           543 non-null    float64
 8   Junction Capacitance CJ(pF)            539 non-null    float64
 9   Peak Pluse Power Dissipation PPPK (W)  542 non-null    float64
 10  Maximum Reverse Leakage IR (uA)        543 non-null    float64
 11  Breakdown Voltage Min 

In [ ]:
# df_selected.head()

,Product,Compliance,Number of Functions,Configuration,Package Type,Reverse Standoff Voltage VRWM(V),Peak Pulse Current IPP(A),Max. Clamping Voltage VC (V),Junction Capacitance CJ(pF),Peak Pluse Power Dissipation PPPK (W),Maximum Reverse Leakage IR (uA),Breakdown Voltage Min VBR(V),Breakdown Voltage Max VBR(V),Junction Temperature Tj [max] (°C),VESDIEC61000-4-2 Air/Contact (kV)
0,MMBZ12VAQ,A R H,2,Unidirectional,SOT-23,12,2.35,17.0,113.00,40.0,0.20,11.4,12.6,150,±30
1,MMBZ5V6AQ,A R H,2,Unidirectional,SOT-23,5.6,3.00,8.0,320.00,24.0,5.00,5.32,5.88,150,±30
2,MMBZ9V1AQ,A R H,2,Unidirectional,SOT-23,9.1,1.70,14.0,183.00,24.0,0.30,8.65,9.56,150,±30
3,ESDSBULC24VLBQ,A R H,1,Bidirectional,DFN1006-2,24,4.00,9.0,0.45,36.0,0.05,25,32,150,±15
4,ESDSBULC18VLBQ,A R H,1,Bidirectional,DFN1006-2,18,4.00,9.0,0.45,36.0,0.05,18.5,NaN,150,±15


In [ ]:
import pandas as pd

df=pd.read_csv(
    'MCC_product_lists/MCC_DataExport_esd-protection-devices(MCC-esd-protection-devices).csv',
    header=0,
    skiprows=[0,2]
)

# 单独去掉ESD0512LB，他有两种参数，如果一起放进df里会直接把float coerce成string
df = df[df['Product'] != 'ESD0512LB']

In [ ]:
def clean_and_prepare_data(df, chip_type):
    """
    清洗数据并整理成统一格式
    """
    df_clean = df.copy()
    
    # 1. 清理列名（去掉换行符、多余空格）
    df_clean.columns = [' '.join(col.split()) for col in df_clean.columns]
    
    # 2. 添加类型列
    df_clean['chip_type'] = chip_type
    
    # 3. 生成 chip_id
    df_clean['chip_id'] = df_clean['Product'] + '_' + chip_type[:2]
    
    return df_clean

In [ ]:
import sqlite3
import os

# ====== 配置每种芯片的参数列 ======
# 每个 CSV 的"专属参数"列（除了 Product、Manufacture、Status、Compliance 这些通用列）
TYPE_PARAMS_CONFIG = {
    'esd-protection-devices': {
        'numeric_cols': [
            'Reverse Standoff Voltage VRWM(V)',
            'Peak Pulse Current IPP(A)',
            'Max. Clamping Voltage VC (V)',
            'Junction Capacitance CJ(pF)',
            'Peak Pluse Power Dissipation PPPK (W)',
            'Maximum Reverse Leakage IR (uA)',
            'Breakdown Voltage Min VBR(V)',
            'Breakdown Voltage Max VBR(V)',
            'Junction Temperature Tj [max] (°C)'
        ],
        'categorical_cols': ['VESDIEC61000-4-2 Air/Contact (kV)']
    },
    # 其他类型继续添加...
}

def import_one_csv(file_path, chip_type, db_path='chip_inventory.db'):
    """
    导入一个 CSV 文件到数据库
    """
    print(f"\n🚀 开始导入: {os.path.basename(file_path)}")
    
    # 3. 清洗数据
    if chip_type in TYPE_PARAMS_CONFIG:
        numeric_cols = TYPE_PARAMS_CONFIG[chip_type]['numeric_cols']
    else:
        numeric_cols = None
    
    df_clean = clean_and_prepare_data(df, chip_type, numeric_cols)
    
    # 4. 连接到数据库
    conn = sqlite3.connect(db_path)
    
    # 5. 写入主表（通用信息）
    # MCC CSV 的通用列：Product, Manufacture, Status, Compliance, Number of Functions, Configuration, Package Type
    common_cols = ['Product', 'Manufacture', 'Status', 'Compliance', 
                   'Number of Functions', 'Configuration', 'Package Type', 'chip_type', 'chip_id']
    
    # 只取存在的列
    available_common = [col for col in common_cols if col in df_clean.columns]
    df_common = df_clean[available_common].copy()
    
    # 重命名列名以符合数据库规范
    df_common = df_common.rename(columns={
        'Product': 'model',
        'Manufacture': 'brand',
        'Package Type': 'package',
        'Number of Functions': 'functions'
    })
    
    # 如果列名不存在，用 None 填充
    if 'model' not in df_common.columns:
        df_common['model'] = df_clean['Product'] if 'Product' in df_clean.columns else None
    if 'brand' not in df_common.columns:
        df_common['brand'] = df_clean['Manufacture'] if 'Manufacture' in df_clean.columns else None
    if 'package' not in df_common.columns:
        df_common['package'] = df_clean['Package Type'] if 'Package Type' in df_clean.columns else None
    if 'functions' not in df_common.columns:
        df_common['functions'] = df_clean['Number of Functions'] if 'Number of Functions' in df_clean.columns else None
    
    # 写入 chips 表
    df_common.to_sql('chips', conn, if_exists='append', index=False)
    print(f"✅ 已写入 {len(df_common)} 条通用信息")
    
    # 6. 写入子表（专属参数）
    if chip_type in TYPE_PARAMS_CONFIG:
        numeric_cols = TYPE_PARAMS_CONFIG[chip_type]['numeric_cols']
        categorical_cols = TYPE_PARAMS_CONFIG[chip_type].get('categorical_cols', [])
        
        # 只取存在的列
        available_numeric = [col for col in numeric_cols if col in df_clean.columns]
        available_categorical = [col for col in categorical_cols if col in df_clean.columns]
        
        df_params = df_clean[['chip_id'] + available_numeric + available_categorical].copy()
        
        # 写入子表
        table_name = f'params_{chip_type}'
        df_params.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"✅ 已写入 {len(df_params)} 条 {chip_type} 专属参数")
    
    conn.close()
    print(f"🎉 {chip_type} 导入完成！")

# ====== 使用示例 ======
# 先测试 ESD 的导入
import_one_csv(
    'MCC_product_lists/MCC_DataExport_esd-protection-devices(MCC-esd-protection-devices).csv',
    'esd-protection-devices'
)